# LeetCode 1959 — Minimum Total Space Wasted With K Resizing Operations bằng Python

Link đề: https://leetcode.com/problems/minimum-total-space-wasted-with-k-resizing-operations/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def minSpaceWastedKResizing(self, nums: List[int], k: int) -> int:
        ...
```

Bài toán thực chất là **chia mảng thành tối đa `k + 1` đoạn liên tiếp**, mỗi đoạn tốn `max(đoạn) · độ_dài − tổng(đoạn)`. Notebook đi từ **thử mọi cách đặt vị trí đổi kích thước** (vét cạn) → **đệ quy có nhớ** → **quy hoạch động dưới lên với bảng chi phí đoạn** → **DP mảng cuộn tính chi phí đoạn ngay trong vòng lặp**.

## 1. Mục tiêu học

1. Dịch đề "mảng động + đổi kích thước" thành bài **phân hoạch mảng** thành các đoạn liên tiếp.
2. Tính **chi phí một đoạn** `[i..j]` = `max · (j − i + 1) − sum` và tiền xử lý mọi đoạn trong `O(n²)`.
3. Thiết kế DP `f[i][j]` = lãng phí nhỏ nhất cho `i` phần tử đầu dùng đúng `j` đoạn; hiểu vì sao "tối đa `k`" tương đương "đúng `k + 1` đoạn".
4. Viết cả bản đệ quy có nhớ và bản dưới lên, tối ưu bộ nhớ bằng mảng cuộn.
5. Đối chiếu với vét cạn trên mảng nhỏ, seed cố định.

## 2. Đề bài & ví dụ

Bạn thiết kế một **mảng động**. Cho mảng `nums` với `nums[t]` là **số phần tử** trong mảng tại thời điểm `t`, và số nguyên `k` là **số lần tối đa** bạn được **đổi kích thước** (sang kích thước tùy ý).

Kích thước tại thời điểm `t`, `size_t`, phải `≥ nums[t]`. **Lãng phí** tại `t` là `size_t − nums[t]`; **tổng lãng phí** là tổng trên mọi `t`. Trả về **tổng lãng phí nhỏ nhất** khi đổi kích thước **tối đa `k` lần**.

Lưu ý: kích thước **ban đầu** tùy ý và **không** tính là một lần đổi.

**Ví dụ 1:** `nums = [10,20]`, `k = 0` → `10` (`size = [20,20]`).

**Ví dụ 2:** `nums = [10,20,30]`, `k = 1` → `10` (`size = [20,20,30]`).

**Ví dụ 3:** `nums = [10,20,15,30,20]`, `k = 2` → `15` (`size = [10,20,20,30,30]`).

**Ràng buộc:** `1 ≤ nums.length ≤ 200`, `1 ≤ nums[i] ≤ 10⁶`, `0 ≤ k ≤ nums.length − 1`.

In [1]:
import random
import time
from functools import lru_cache
from itertools import combinations
from typing import List

INF = float('inf')
examples = [([10, 20], 0, 10), ([10, 20, 30], 1, 10), ([10, 20, 15, 30, 20], 2, 15)]

def waste_of_sizes(nums, sizes):
    assert all(s >= x for s, x in zip(sizes, nums))
    resizes = sum(sizes[t] != sizes[t - 1] for t in range(1, len(sizes)))
    return sum(s - x for s, x in zip(sizes, nums)), resizes

print(waste_of_sizes([10, 20], [20, 20]))
print(waste_of_sizes([10, 20, 30], [20, 20, 30]))
print(waste_of_sizes([10, 20, 15, 30, 20], [10, 20, 20, 30, 30]), "<- (lãng phí, số lần đổi)")

(10, 0)
(10, 1)
(15, 2) <- (lãng phí, số lần đổi)


## 3. Kiến thức nền

### 3.1 Từ "đổi kích thước" sang "chia đoạn"

Giữa hai lần đổi, kích thước **không đổi** ⇒ thời gian bị chia thành các **đoạn liên tiếp**, mỗi đoạn dùng một kích thước. Với `r` lần đổi ta có `r + 1` đoạn. Trong một đoạn, kích thước tốt nhất là **nhỏ nhất có thể** = `max` của đoạn.

In [2]:
def seg_cost(nums, i, j):
    """Chi phí đoạn nums[i..j] (gồm hai đầu) khi dùng kích thước = max đoạn."""
    seg = nums[i:j + 1]
    return max(seg) * len(seg) - sum(seg)

nums = [10, 20, 15, 30, 20]
print("đoạn [10,20,15]:", seg_cost(nums, 0, 2), "| đoạn [30,20]:", seg_cost(nums, 3, 4), "| cả mảng:", seg_cost(nums, 0, 4))

đoạn [10,20,15]: 15 | đoạn [30,20]: 10 | cả mảng: 55


### 3.2 Tính chi phí **mọi** đoạn trong `O(n²)`

Cố định điểm đầu `i`, mở rộng `j` dần, cập nhật `max` và `sum` chạy ⇒ mỗi đoạn `O(1)`.

In [3]:
def all_seg_costs(nums):
    n = len(nums)
    g = [[0] * n for _ in range(n)]
    for i in range(n):
        mx = s = 0
        for j in range(i, n):
            mx = max(mx, nums[j]); s += nums[j]
            g[i][j] = mx * (j - i + 1) - s
    return g

g = all_seg_costs(nums)
for i in range(len(nums)):
    print("   ", [g[i][j] if j >= i else '-' for j in range(len(nums))])

    [0, 10, 15, 45, 55]
    ['-', 0, 5, 25, 35]
    ['-', '-', 0, 15, 25]
    ['-', '-', '-', 0, 10]
    ['-', '-', '-', '-', 0]


### 3.3 `itertools.combinations` để chọn vị trí cắt

Chọn `r` vị trí cắt trong `n − 1` khe giữa các phần tử: `combinations(range(1, n), r)`.

In [4]:
print(list(combinations(range(1, 5), 2)))

[(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]


### 3.4 Vô cực `float('inf')` trong DP tối thiểu

Khởi tạo mọi trạng thái là `inf` ("chưa đạt được"), chỉ trạng thái gốc là `0`. `min(inf, x) = x`, `inf + x = inf`.

In [5]:
print(min(INF, 5), INF + 3, INF > 10**18)

5 inf True


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 3: `nums = [10,20,15,30,20]`, `k = 2` ⇒ tối đa 3 đoạn. Thử tay vài cách chia:

| cách chia | chi phí từng đoạn | tổng |
|---|---|---|
| `[10] [20,15] [30,20]` | 0 + 5 + 10 | **15** |
| `[10,20] [15] [30,20]` | 10 + 0 + 10 | 20 |
| `[10,20,15] [30] [20]` | 15 + 0 + 0 | 15 |
| `[10] [20,15,30] [20]` | 0 + 25 + 0 | 25 |
| `[10,20,15,30,20]` (1 đoạn) | 65 | 65 |

Nhận xét: **cắt thêm không bao giờ làm tệ hơn** (đoạn con có `max` nhỏ hơn hoặc bằng), nên "tối đa `k` lần" = "đúng `k + 1` đoạn" (vì `k + 1 ≤ n`). Bài toán thành: chia thành đúng `k + 1` đoạn, tối thiểu tổng chi phí — bài **DP phân hoạch** kinh điển.

In [6]:
def cost_of_cuts(nums, cuts):
    bounds = [0, *cuts, len(nums)]
    return sum(seg_cost(nums, bounds[t], bounds[t + 1] - 1) for t in range(len(bounds) - 1))

for cuts in [(1, 3), (2, 3), (3, 4), (1, 4), ()]:
    bounds = [0, *cuts, 5]
    parts = [nums[bounds[t]:bounds[t + 1]] for t in range(len(bounds) - 1)]
    print(f"cắt tại {cuts}: {parts} -> {cost_of_cuts(nums, cuts)}")

cắt tại (1, 3): [[10], [20, 15], [30, 20]] -> 15
cắt tại (2, 3): [[10, 20], [15], [30, 20]] -> 20
cắt tại (3, 4): [[10, 20, 15], [30], [20]] -> 15
cắt tại (1, 4): [[10], [20, 15, 30], [20]] -> 25
cắt tại (): [[10, 20, 15, 30, 20]] -> 55


## 5. Cách 1 — Brute Force / baseline: thử mọi tập vị trí cắt

**Ý tưởng:** với `r = 0..k` lần đổi, thử mọi tập `r` vị trí cắt trong `n − 1` khe; tính tổng chi phí; lấy nhỏ nhất.

In [7]:
def min_waste_brute(nums, k):
    n = len(nums)
    best = INF
    for r in range(0, k + 1):
        for cuts in combinations(range(1, n), r):
            best = min(best, cost_of_cuts(nums, cuts))
    return best

for nums_, k_, exp in examples:
    print(min_waste_brute(nums_, k_), "(kỳ vọng", exp, ")")

10 (kỳ vọng 10 )
10 (kỳ vọng 10 )
15 (kỳ vọng 15 )


**Độ phức tạp:** `Σ_{r ≤ k} C(n − 1, r)` tập cắt, mỗi tập `O(n)` ⇒ tới `O(2ⁿ · n)`. Với `n = 200` bất khả thi.

**Nút thắt:** chi phí phần **sau** một vị trí cắt chỉ phụ thuộc **vị trí cắt** và **số lần đổi còn lại** — không phụ thuộc cách cắt phía trước ⇒ **bài toán con chồng nhau** ⇒ quy hoạch động.

## 6. Ý tưởng thuật toán / bước đột phá

**Trạng thái:** `f[i][j]` = tổng lãng phí nhỏ nhất cho `i` phần tử đầu (`nums[0..i−1]`) khi chia thành **đúng** `j` đoạn.

**Chuyển:** đoạn cuối là `nums[h..i−1]` với `j − 1 ≤ h < i`:
`f[i][j] = min_h ( f[h][j − 1] + g[h][i − 1] )`, với `f[0][0] = 0`.

**Đáp án:** `f[n][k + 1]` (hoặc `min_{j ≤ k+1} f[n][j]` — bằng nhau vì cắt thêm không làm tệ hơn).

**Bất biến:** khi tính hàng `j`, mọi `f[·][j − 1]` đã là giá trị tối ưu.

**Đệ quy tương đương:** `dfs(i, r)` = lãng phí nhỏ nhất cho hậu tố `nums[i..]` khi còn `r` lần đổi: thử mọi điểm kết thúc `e` của đoạn hiện tại: `cost(i, e) + dfs(e + 1, r − 1)` (chỉ khi `r > 0` hoặc `e = n − 1`).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đệ quy có nhớ `dfs(i, r)`

In [8]:
def min_waste_memo(nums, k):
    n = len(nums)
    @lru_cache(maxsize=None)
    def dfs(i, r):
        if i == n:
            return 0
        best = INF
        mx = s = 0
        for e in range(i, n):              # đoạn hiện tại nums[i..e]
            mx = max(mx, nums[e]); s += nums[e]
            cost = mx * (e - i + 1) - s
            if e == n - 1:
                best = min(best, cost)     # đoạn cuối cùng
            elif r > 0:
                best = min(best, cost + dfs(e + 1, r - 1))
        return best
    ans = dfs(0, k)
    dfs.cache_clear()
    return ans

for nums_, k_, exp in examples:
    print(min_waste_memo(nums_, k_), "(kỳ vọng", exp, ")")

10 (kỳ vọng 10 )
10 (kỳ vọng 10 )
15 (kỳ vọng 15 )


**Độ phức tạp:** `O(n · k)` trạng thái × `O(n)` mỗi trạng thái ⇒ `O(n² k)`; độ sâu đệ quy `≤ k + 1 ≤ 200` — an toàn. Bộ nhớ `O(n k)`.

### 7.2 Cách 3 — DP dưới lên với bảng chi phí đoạn `g`

In [9]:
def min_waste_table(nums, k):
    n = len(nums)
    g = all_seg_costs(nums)
    K = k + 1
    f = [[INF] * (K + 1) for _ in range(n + 1)]
    f[0][0] = 0
    for i in range(1, n + 1):
        for j in range(1, min(i, K) + 1):
            best = INF
            for h in range(j - 1, i):
                v = f[h][j - 1] + g[h][i - 1]
                if v < best:
                    best = v
            f[i][j] = best
    return f[n][K]

class SolutionTable:
    def minSpaceWastedKResizing(self, nums: List[int], k: int) -> int:
        n, K = len(nums), k + 1
        g = [[0] * n for _ in range(n)]
        for i in range(n):
            mx = s = 0
            for j in range(i, n):
                mx = max(mx, nums[j]); s += nums[j]
                g[i][j] = mx * (j - i + 1) - s
        f = [[INF] * (K + 1) for _ in range(n + 1)]
        f[0][0] = 0
        for i in range(1, n + 1):
            for j in range(1, min(i, K) + 1):
                f[i][j] = min(f[h][j - 1] + g[h][i - 1] for h in range(j - 1, i))
        return f[n][K]

for nums_, k_, exp in examples:
    print(min_waste_table(nums_, k_), SolutionTable().minSpaceWastedKResizing(nums_, k_), "(kỳ vọng", exp, ")")

10 10 (kỳ vọng 10 )
10 10 (kỳ vọng 10 )
15 15 (kỳ vọng 15 )


Giới hạn `h ≥ j − 1` vì `j − 1` đoạn đầu cần ít nhất `j − 1` phần tử; `j ≤ i` vì mỗi đoạn có ít nhất một phần tử. **Độ phức tạp:** `O(n² k)` thời gian, `O(n²)` cho `g` + `O(n k)` cho `f`.

### 7.3 Cách 4 — DP mảng cuộn, tính chi phí đoạn ngay khi quét ngược (lời giải chính)

Duyệt `j` (số đoạn) ở vòng ngoài, giữ `prev = f[·][j − 1]`, `cur = f[·][j]`. Với mỗi `i`, quét `h` **từ `i − 1` lùi về** để cập nhật `max`/`sum` của đoạn `nums[h..i−1]` trong `O(1)` — không cần bảng `g` ⇒ bộ nhớ `O(n)`.

In [10]:
def min_waste_rolling(nums, k):
    n, K = len(nums), k + 1
    prev = [INF] * (n + 1)
    prev[0] = 0                                    # f[·][0]
    for j in range(1, K + 1):
        cur = [INF] * (n + 1)
        for i in range(j, n + 1):
            mx = s = 0
            best = INF
            for h in range(i - 1, j - 2, -1):      # đoạn cuối nums[h..i-1], h >= j-1
                x = nums[h]
                if x > mx: mx = x
                s += x
                v = prev[h] + mx * (i - h) - s
                if v < best: best = v
            cur[i] = best
        prev = cur
    return prev[n]

for nums_, k_, exp in examples:
    print(min_waste_rolling(nums_, k_), "(kỳ vọng", exp, ")")

10 (kỳ vọng 10 )
10 (kỳ vọng 10 )
15 (kỳ vọng 15 )


**Độ phức tạp:** `O(n² k)` thời gian, `O(n)` bộ nhớ.

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 3 (`nums = [10,20,15,30,20]`, `K = 3`): in bảng `f[i][j]` và với mỗi ô, điểm cắt `h` tốt nhất.

In [11]:
def trace_table(nums, k):
    n, K = len(nums), k + 1
    g = all_seg_costs(nums)
    f = [[INF] * (K + 1) for _ in range(n + 1)]
    arg = [[None] * (K + 1) for _ in range(n + 1)]
    f[0][0] = 0
    for i in range(1, n + 1):
        for j in range(1, min(i, K) + 1):
            for h in range(j - 1, i):
                v = f[h][j - 1] + g[h][i - 1]
                if v < f[i][j]:
                    f[i][j], arg[i][j] = v, h
            print(f"  f[{i}][{j}] = {f[i][j]:>3}  (đoạn cuối nums[{arg[i][j]}..{i - 1}] = {nums[arg[i][j]:i]}, chi phí {g[arg[i][j]][i - 1]})")
    # truy vết
    parts, i, j = [], n, K
    while i > 0:
        h = arg[i][j]; parts.append(nums[h:i]); i, j = h, j - 1
    print("  chia tối ưu:", parts[::-1], "-> tổng", f[n][K])
    return f[n][K]

trace_table([10, 20, 15, 30, 20], 2)

  f[1][1] =   0  (đoạn cuối nums[0..0] = [10], chi phí 0)
  f[2][1] =  10  (đoạn cuối nums[0..1] = [10, 20], chi phí 10)
  f[2][2] =   0  (đoạn cuối nums[1..1] = [20], chi phí 0)
  f[3][1] =  15  (đoạn cuối nums[0..2] = [10, 20, 15], chi phí 15)
  f[3][2] =   5  (đoạn cuối nums[1..2] = [20, 15], chi phí 5)
  f[3][3] =   0  (đoạn cuối nums[2..2] = [15], chi phí 0)
  f[4][1] =  45  (đoạn cuối nums[0..3] = [10, 20, 15, 30], chi phí 45)
  f[4][2] =  15  (đoạn cuối nums[3..3] = [30], chi phí 0)
  f[4][3] =   5  (đoạn cuối nums[3..3] = [30], chi phí 0)
  f[5][1] =  55  (đoạn cuối nums[0..4] = [10, 20, 15, 30, 20], chi phí 55)
  f[5][2] =  25  (đoạn cuối nums[3..4] = [30, 20], chi phí 10)
  f[5][3] =  15  (đoạn cuối nums[3..4] = [30, 20], chi phí 10)
  chia tối ưu: [[10], [20, 15], [30, 20]] -> tổng 15


15

Trace cách 2 (đệ quy) trên ví dụ 2 — in các trạng thái được tính lần đầu:

In [12]:
def trace_memo(nums, k):
    n = len(nums); memo = {}
    def dfs(i, r, d=0):
        if i == n: return 0
        if (i, r) in memo: return memo[(i, r)]
        best, choice = INF, None
        mx = s = 0
        for e in range(i, n):
            mx = max(mx, nums[e]); s += nums[e]
            cost = mx * (e - i + 1) - s
            if e == n - 1:
                cand = cost
            elif r > 0:
                cand = cost + dfs(e + 1, r - 1, d + 1)
            else:
                continue
            if cand < best:
                best, choice = cand, e
        memo[(i, r)] = best
        print("  " * d + f"dfs(i={i}, r={r}) = {best}  (đoạn đầu nums[{i}..{choice}] = {nums[i:choice + 1]})")
        return best
    return dfs(0, k)

trace_memo([10, 20, 30], 1)

  dfs(i=1, r=0) = 10  (đoạn đầu nums[1..2] = [20, 30])
  dfs(i=2, r=0) = 0  (đoạn đầu nums[2..2] = [30])
dfs(i=0, r=1) = 10  (đoạn đầu nums[0..0] = [10])


10

## 9. Code nộp LeetCode

In [13]:
class Solution:
    def minSpaceWastedKResizing(self, nums: List[int], k: int) -> int:
        n, K = len(nums), k + 1                  # tối đa k lần đổi = tối đa k+1 đoạn
        INF = float('inf')
        prev = [INF] * (n + 1)
        prev[0] = 0
        for j in range(1, K + 1):
            cur = [INF] * (n + 1)
            for i in range(j, n + 1):
                mx = s = 0
                best = INF
                for h in range(i - 1, j - 2, -1):   # đoạn cuối nums[h..i-1]
                    x = nums[h]
                    if x > mx:
                        mx = x
                    s += x
                    v = prev[h] + mx * (i - h) - s
                    if v < best:
                        best = v
                cur[i] = best
            prev = cur
        return prev[n]

for nums_, k_, exp in examples:
    assert Solution().minSpaceWastedKResizing(nums_, k_) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Quy đổi.** Một lịch kích thước với `r ≤ k` lần đổi chia `[0, n)` thành `r + 1` đoạn liên tiếp có kích thước không đổi; trong đoạn `S`, kích thước phải `≥ max(S)`, lãng phí `≥ max(S)·|S| − sum(S)`, đạt dấu bằng khi chọn đúng `max(S)`. Ngược lại mọi cách chia `≤ k + 1` đoạn cho một lịch hợp lệ. ⇒ đáp án = chi phí nhỏ nhất khi chia thành `≤ k + 1` đoạn.

**"Tối đa" = "đúng" `k + 1` đoạn.** Tách một đoạn `S` thành `S₁S₂` không làm tăng chi phí: `max(Sᵢ) ≤ max(S)`. Vì `k + 1 ≤ n`, từ một cách chia ít đoạn hơn luôn tách thêm được tới đúng `k + 1` đoạn mà không tệ hơn.

**Công thức DP.** Trong cách chia tối ưu `nums[0..i−1]` thành `j` đoạn, đoạn cuối bắt đầu ở `h`; phần trước là một cách chia `j − 1` đoạn của `nums[0..h−1]`, và nó phải tối ưu (nếu không, thay bằng cách tốt hơn sẽ giảm tổng) — **cấu trúc con tối ưu**. Lấy `min` trên mọi `h` cho đúng `f[i][j]`.

**Cách 4** chỉ thay đổi thứ tự tính `g[h][i−1]` (quét `h` lùi, cập nhật `max`, `sum` chạy) — cùng công thức.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `K = k + 1`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn tập cắt | `O(2ⁿ · n)` | `O(n)` |
| 2. Đệ quy có nhớ | `O(n² K)` | `O(n K)` |
| 3. Bảng `g` + `f` | `O(n² K)` | `O(n² + n K)` |
| 4. Mảng cuộn | `O(n² K)` | `O(n)` |

Với `n = 200`, `K = 200`: khoảng `n²K/2 ≈ 4·10⁶` phép tính lõi — vài giây trong Python thuần. (Có thể tối ưu thêm vì `f[i][j]` chỉ cần `i ≥ j`.) Đo thời gian với `n = 200`, `k = 20`:

In [14]:
rnd = random.Random(1959)
big = [rnd.randint(1, 10**6) for _ in range(200)]
for name, fn in [("memo", min_waste_memo), ("table", min_waste_table), ("rolling", min_waste_rolling), ("Solution", Solution().minSpaceWastedKResizing)]:
    t0 = time.perf_counter(); r = fn(big, 20); dt = time.perf_counter() - t0
    print(f"{name:<9} n=200, k=20: {r:>12}  {dt * 1000:7.0f} ms")

memo      n=200, k=20:     64067643       63 ms
table     n=200, k=20:     64067643       22 ms
rolling   n=200, k=20:     64067643       26 ms
Solution  n=200, k=20:     64067643       26 ms


## 12. Bộ test

In [15]:
ALL = {
    "memo": min_waste_memo,
    "table": min_waste_table,
    "rolling": min_waste_rolling,
    "SolTable": lambda a, k: SolutionTable().minSpaceWastedKResizing(a, k),
    "Solution": lambda a, k: Solution().minSpaceWastedKResizing(a, k),
}

def check(nums, k, exp=None, brute=True):
    ref = min_waste_brute(nums, k) if brute else min_waste_table(nums, k)
    for name, fn in ALL.items():
        got = fn(list(nums), k)
        assert got == ref, (name, nums, k, got, ref)
    if exp is not None:
        assert ref == exp, (nums, k, ref, exp)
    return ref

for nums_, k_, exp in examples:
    check(nums_, k_, exp)
print("Ví dụ đề: OK")

check([5], 0, 0)                          # một phần tử
check([3, 1, 4, 1, 5], 4, 0)              # k = n - 1: mỗi phần tử một đoạn
check([7, 7, 7, 7], 0, 0)                 # bằng nhau
check([1, 100], 0, 99)
check([1, 100, 1, 100], 1, 99)            # tự dựng: [1] [100,1,100] -> 0 + 99
check([5, 4, 3, 2, 1], 1, 4)              # tự dựng: [5,4] [3,2,1] -> 1 + 3
print("Edge case: OK")

Ví dụ đề: OK
Edge case: OK


Hai edge case cuối là **tự dựng** — kỳ vọng được brute force xác nhận ngay trong `check` (đối chiếu lại đề gốc nếu muốn). Ví dụ `[5,4,3,2,1]`, `k = 1`: cách tốt nhất là `[5,4] [3,2,1]` với chi phí `1 + 3 = 4` (không phải `[5] [4,3,2,1]` = 6).

In [16]:
rnd = random.Random(2009)
for _ in range(1500):
    n = rnd.randint(1, 9)
    nums_ = [rnd.randint(1, rnd.choice([5, 50, 10**6])) for _ in range(n)]
    check(nums_, rnd.randint(0, n - 1))
print("1500 mảng ngẫu nhiên nhỏ so với vét cạn: OK")

for _ in range(10):
    n = rnd.randint(20, 60)
    nums_ = [rnd.randint(1, 10**6) for _ in range(n)]
    check(nums_, rnd.randint(0, n - 1), brute=False)
print("10 mảng cỡ vừa so giữa các cách DP: OK")

1500 mảng ngẫu nhiên nhỏ so với vét cạn: OK
10 mảng cỡ vừa so giữa các cách DP: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Coi `k` là số đoạn (quên `+1`)

In [17]:
def min_waste_bug_segments(nums, k):
    return min_waste_table(nums, k - 1) if k >= 1 else min_waste_table(nums, 0)   # SAI: dùng k đoạn thay vì k+1

nums_, k_ = [10, 20, 30], 1
print("sai :", min_waste_bug_segments(nums_, k_), "| đúng:", Solution().minSpaceWastedKResizing(nums_, k_))

sai : 30 | đúng: 10


**Vì sao sai:** kích thước **ban đầu** là miễn phí; `k` lần đổi tạo ra `k + 1` đoạn. **Sửa:** `K = k + 1`.

### Lỗi 2 — Chi phí đoạn dùng `max` của **cả mảng**

In [18]:
def min_waste_bug_global_max(nums, k):
    M = max(nums)
    return sum(M - x for x in nums)             # SAI: bỏ qua việc đổi kích thước

nums_, k_ = [10, 20, 15, 30, 20], 2
print("sai :", min_waste_bug_global_max(nums_, k_), "| đúng:", Solution().minSpaceWastedKResizing(nums_, k_))

sai : 55 | đúng: 15


**Sửa:** mỗi đoạn dùng `max` **của chính đoạn đó**: `max(seg)·len(seg) − sum(seg)`.

### Lỗi 3 — Tham lam: cắt tại các "bước nhảy" lớn nhất

In [19]:
def min_waste_bug_greedy(nums, k):
    n = len(nums)
    jumps = sorted(range(1, n), key=lambda t: -(abs(nums[t] - nums[t - 1])))[:k]   # cắt ở k chênh lệch lớn nhất
    return cost_of_cuts(nums, sorted(jumps))

rnd = random.Random(3)
found = None
while found is None:
    a = [rnd.randint(1, 20) for _ in range(rnd.randint(3, 7))]
    kk = rnd.randint(1, len(a) - 1)
    if min_waste_bug_greedy(a, kk) != Solution().minSpaceWastedKResizing(a, kk):
        found = (a, kk)
print("phản ví dụ:", found, "| tham lam:", min_waste_bug_greedy(*found), "| đúng:", Solution().minSpaceWastedKResizing(*found))

phản ví dụ: ([16, 3, 12, 3, 14, 5], 1) | tham lam: 33 | đúng: 32


**Vì sao sai:** chi phí đoạn phụ thuộc vào **max và độ dài** của cả đoạn, không chỉ chênh lệch tại chỗ cắt. **Sửa:** quy hoạch động xét mọi điểm cắt.

### Lỗi 4 — Khởi tạo `f` bằng 0 thay vì `inf`

In [20]:
def min_waste_bug_init0(nums, k):
    n, K = len(nums), k + 1
    g = all_seg_costs(nums)
    f = [[0] * (K + 1) for _ in range(n + 1)]            # SAI: trạng thái không đạt được lại có giá trị 0
    for i in range(1, n + 1):
        for j in range(1, K + 1):
            f[i][j] = min(f[h][j - 1] + g[h][i - 1] for h in range(0, i))
    return f[n][K]

nums_, k_ = [10, 20, 15, 30, 20], 2
print("sai :", min_waste_bug_init0(nums_, k_), "| đúng:", Solution().minSpaceWastedKResizing(nums_, k_))

sai : 0 | đúng: 15


**Vì sao sai:** `f[h][0] = 0` với `h > 0` nghĩa là "chia `h` phần tử thành **0 đoạn** với chi phí 0" — vô lý, làm DP "bỏ qua" phần đầu mảng. **Sửa:** `f = inf`, chỉ `f[0][0] = 0`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chia mảng thành tối đa/đúng `k` đoạn liên tiếp, chi phí mỗi đoạn tính được" ⇒ **DP phân hoạch** `f[i][j] = min_h f[h][j−1] + cost(h, i−1)`, `O(n² k)`.
- Chi phí đoạn dạng `max·len − sum` ⇒ tính bằng cách mở rộng/quét ngược với `max`, `sum` chạy.
- Đọc kỹ "ban đầu không tính" ⇒ số đoạn = số lần đổi + 1.
- Nếu tách đoạn không bao giờ làm tệ hơn ⇒ "tối đa `k`" = "đúng `k`" (khi `k ≤ n`).
- Brute force tập cắt (`combinations`) cực kỳ hữu ích để kiểm DP.

## 15. Biến thể của bài

1. **LeetCode 410 — Split Array Largest Sum:** chia `k` đoạn, tối thiểu **tổng lớn nhất** — DP `O(n²k)` hoặc tìm kiếm nhị phân + tham lam.
2. **LeetCode 1335 — Minimum Difficulty of a Job Schedule:** chia `d` đoạn, chi phí đoạn là `max` — cùng khung DP.
3. **LeetCode 813 — Largest Sum of Averages:** chia tối đa `k` đoạn, tối đa tổng trung bình.
4. **Chi phí đổi kích thước khác 0** (mỗi lần đổi tốn `c`): `f` thêm `+c` khi mở đoạn mới; bỏ ràng buộc `k`.
5. **Trả về chính lịch kích thước:** truy vết `arg` như mục 8.

In [21]:
def min_difficulty(jobs, d):
    """LeetCode 1335 cùng khung DP: chia đúng d đoạn, chi phí đoạn = max."""
    n = len(jobs)
    if n < d: return -1
    prev = [INF] * (n + 1); prev[0] = 0
    for j in range(1, d + 1):
        cur = [INF] * (n + 1)
        for i in range(j, n + 1):
            mx, best = 0, INF
            for h in range(i - 1, j - 2, -1):
                mx = max(mx, jobs[h]); best = min(best, prev[h] + mx)
            cur[i] = best
        prev = cur
    return prev[n]

def min_waste_with_resize_cost(nums, c):
    """Biến thể 4: không giới hạn số lần đổi, mỗi lần tốn c."""
    n = len(nums)
    f = [0] + [INF] * n
    for i in range(1, n + 1):
        mx = s = 0
        for h in range(i - 1, -1, -1):
            mx = max(mx, nums[h]); s += nums[h]
            f[i] = min(f[i], f[h] + mx * (i - h) - s + (c if h > 0 else 0))
    return f[n]

print("1335 [6,5,4,3,2,1], d=2 ->", min_difficulty([6, 5, 4, 3, 2, 1], 2))
print("đổi tốn c=4, [10,20,15,30,20] ->", min_waste_with_resize_cost([10, 20, 15, 30, 20], 4))

1335 [6,5,4,3,2,1], d=2 -> 7
đổi tốn c=4, [10,20,15,30,20] -> 16


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | mọi tập vị trí cắt | `O(2ⁿ n)` | `O(n)` | kiểm thử `n ≤ 12` |
| 2. Đệ quy có nhớ | `dfs(i, r)` trên hậu tố | `O(n² K)` | `O(n K)` | dễ nghĩ từ trên xuống |
| 3. Bảng `g`, `f` | DP phân hoạch | `O(n² K)` | `O(n²)` | dễ truy vết |
| 4. Mảng cuộn | quét `h` lùi, `max/sum` chạy | `O(n² K)` | `O(n)` | **nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().minSpaceWastedKResizing([1, 2, 3, 4], 0)` và `([1, 2, 3, 4], 3)`?

**Bài 2 (tính tay):** Lập bảng `g[i][j]` (chi phí mọi đoạn) cho `nums = [4, 1, 3, 2]`, rồi tính đáp án với `k = 1`.

**Bài 3 (điền TODO):** Hoàn thiện `best_sizes(nums, k)` trả về **mảng kích thước** `size[t]` tối ưu (đúng tổng lãng phí nhỏ nhất, số lần đổi ≤ k).

**Bài 4 (tự viết lại):** Viết lại cách 2 (đệ quy) thành DP dưới lên **theo hậu tố** `h[i][r]` (lãng phí nhỏ nhất của `nums[i..]` với `r` lần đổi).

**Bài 5 (mini challenge):** Biến thể "mỗi lần đổi kích thước tốn `c`, không giới hạn số lần": viết `min_total_cost(nums, c)` và kiểm bằng vét cạn mọi tập cắt.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def best_sizes(nums, k):
    # TODO: DP bảng như trace_table, lưu arg, truy vết các đoạn; size của đoạn = max đoạn
    pass

# Bài 4 — TODO
def min_waste_suffix_dp(nums, k):
    # TODO: h[n][r] = 0; h[i][r] = min_e cost(i, e) + h[e+1][r-1] (r>0 hoặc e = n-1)
    pass

# Bài 5 — TODO
def min_total_cost(nums, c):
    # TODO: f[i] = min_h f[h] + cost(h, i-1) + (c nếu h > 0)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `k = 0` ⇒ một đoạn, kích thước `4`.
- Mức 2: `3 + 2 + 1 + 0 = 6`.
- Mức 3: `k = 3` ⇒ mỗi phần tử một đoạn ⇒ `0`.

**Bài 2**
- Mức 1: `g[0][1] = 4·2 − 5 = 3`.
- Mức 2: Với `k = 1` thử 3 vị trí cắt.
- Mức 3: `[4,1] [3,2]` ⇒ `3 + 1 = 4`; `[4] [1,3,2]` ⇒ `0 + 3 = 3`.

**Bài 3**
- Mức 1: Truy vết `arg[i][j]` từ `(n, K)` về `(0, 0)`.
- Mức 2: Mỗi đoạn `nums[h..i−1]` có kích thước `max`.
- Mức 3: Kiểm bằng `waste_of_sizes`.

**Bài 4**
- Mức 1: Duyệt `i` từ `n − 1` về `0`.
- Mức 2: `r` từ `0` tới `k`.
- Mức 3: Đáp án `h[0][k]`.

**Bài 5**
- Mức 1: Không còn chiều `j`.
- Mức 2: Đoạn đầu không tốn `c` (kích thước ban đầu miễn phí).
- Mức 3: Brute force: thử mọi tập cắt, cộng `c · số_cắt`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1
r1 = (Solution().minSpaceWastedKResizing([1, 2, 3, 4], 0), Solution().minSpaceWastedKResizing([1, 2, 3, 4], 3))
print("Bài 1:", r1)
assert r1 == (6, 0)

# Bài 2
g2 = all_seg_costs([4, 1, 3, 2])
for row in g2:
    print("  ", row)
assert Solution().minSpaceWastedKResizing([4, 1, 3, 2], 1) == 3 == min_waste_brute([4, 1, 3, 2], 1)
print("  Bài 2: đáp án k=1 ->", Solution().minSpaceWastedKResizing([4, 1, 3, 2], 1))

# Bài 3
def best_sizes_sol(nums, k):
    n, K = len(nums), k + 1
    g = all_seg_costs(nums)
    f = [[INF] * (K + 1) for _ in range(n + 1)]
    arg = [[None] * (K + 1) for _ in range(n + 1)]
    f[0][0] = 0
    for i in range(1, n + 1):
        for j in range(1, min(i, K) + 1):
            for h in range(j - 1, i):
                v = f[h][j - 1] + g[h][i - 1]
                if v < f[i][j]:
                    f[i][j], arg[i][j] = v, h
    sizes = [0] * n
    i, j = n, K
    while i > 0:
        h = arg[i][j]
        m = max(nums[h:i])
        for t in range(h, i):
            sizes[t] = m
        i, j = h, j - 1
    return sizes

rnd = random.Random(3)
for _ in range(500):
    a = [rnd.randint(1, 30) for _ in range(rnd.randint(1, 10))]
    kk = rnd.randint(0, len(a) - 1)
    w, rs = waste_of_sizes(a, best_sizes_sol(a, kk))
    assert w == Solution().minSpaceWastedKResizing(a, kk) and rs <= kk
print("Bài 3 OK | ví dụ 3:", best_sizes_sol([10, 20, 15, 30, 20], 2))

Bài 1: (6, 0)
   [0, 3, 4, 6]
   [0, 0, 2, 3]
   [0, 0, 0, 1]
   [0, 0, 0, 0]
  Bài 2: đáp án k=1 -> 3
Bài 3 OK | ví dụ 3: [10, 20, 20, 30, 30]


In [24]:
# Bài 4
def min_waste_suffix_dp_sol(nums, k):
    n = len(nums)
    h = [[INF] * (k + 1) for _ in range(n + 1)]
    for r in range(k + 1):
        h[n][r] = 0
    for i in range(n - 1, -1, -1):
        for r in range(k + 1):
            mx = s = 0
            best = INF
            for e in range(i, n):
                mx = max(mx, nums[e]); s += nums[e]
                cost = mx * (e - i + 1) - s
                if e == n - 1:
                    best = min(best, cost)
                elif r > 0:
                    best = min(best, cost + h[e + 1][r - 1])
            h[i][r] = best
    return h[0][k]

# Bài 5
def min_total_cost_sol(nums, c):
    return min_waste_with_resize_cost(nums, c)

def min_total_cost_brute(nums, c):
    n = len(nums)
    return min(cost_of_cuts(nums, cuts) + c * len(cuts) for r in range(n) for cuts in combinations(range(1, n), r))

rnd = random.Random(5)
for _ in range(600):
    a = [rnd.randint(1, 30) for _ in range(rnd.randint(1, 9))]
    kk = rnd.randint(0, len(a) - 1)
    assert min_waste_suffix_dp_sol(a, kk) == Solution().minSpaceWastedKResizing(a, kk)
    cc = rnd.randint(0, 20)
    assert min_total_cost_sol(a, cc) == min_total_cost_brute(a, cc)
print("Bài 4, Bài 5 OK | c=4 với ví dụ 3 ->", min_total_cost_sol([10, 20, 15, 30, 20], 4))

Bài 4, Bài 5 OK | c=4 với ví dụ 3 -> 16


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template DP phân hoạch "đúng K đoạn" (mảng cuộn):**

```python
prev = [INF] * (n + 1); prev[0] = 0
for j in range(1, K + 1):
    cur = [INF] * (n + 1)
    for i in range(j, n + 1):
        # quét đoạn cuối nums[h..i-1] từ h = i-1 lùi về j-1, cập nhật thông tin đoạn
        for h in range(i - 1, j - 2, -1):
            cur[i] = min(cur[i], prev[h] + cost(h, i - 1))
    prev = cur
return prev[n]
```

**Bài này:** `K = k + 1`, `cost = max·len − sum` (cập nhật `max`, `sum` khi lùi `h`).

**Checklist:**
- [ ] Số đoạn `= k + 1` (kích thước ban đầu miễn phí)?
- [ ] Chi phí đoạn dùng `max` **của đoạn**?
- [ ] Khởi tạo `inf`, chỉ `f[0][0] = 0`?
- [ ] Giới hạn `h ≥ j − 1` (mỗi đoạn ≥ 1 phần tử)?
- [ ] Đã so với vét cạn tập cắt trên mảng nhỏ?
- [ ] Chữ ký đúng: `def minSpaceWastedKResizing(self, nums: List[int], k: int) -> int`?